In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten/o4_mini_high_small_q5/checkpoints/post_cell_37_try_0.pickle

In [ ]:
%%cudf.pandas.profile
### cell 38 ###

def grab_subset_of_data_50(original_df, question_of_interest):
    # 1) build a GPU‐accelerated boolean mask over column names
    mask = original_df.columns.to_series().str.contains(question_of_interest)
    # 2) use .loc with the boolean mask to pick matching columns on GPU
    subset_df = original_df.loc[:, mask]

    # 3) GPU‐vectorized rename: split on '-' and strip whitespace
    new_names = (
        subset_df.columns.to_series()
                 .str.split('-')
                 .str.get(-1)
                 .str.lstrip()
                 .to_list()  # final list conversion
    )
    subset_df.columns = new_names

    # 4) drop any rows where all values across these columns are null (GPU)
    return subset_df.dropna(how='all')
